In [6]:
import pandas as pd
import glob

files = sorted(glob.glob("../data/raw/**/*.csv", recursive=True))

print(len(files), "files")
print(files[0])

91 files
../data/raw\data_Q2_2026\2026-04-01.csv


In [7]:
import glob, pandas as pd
from collections import Counter

files = sorted(glob.glob("../data/raw/**/*.csv", recursive=True))
print(len(files), "files")

drive_days, fails = Counter(), Counter()
for f in files:
    d = pd.read_csv(f, usecols=["model", "failure"])
    drive_days.update(d["model"].value_counts().to_dict())
    fails.update(d.loc[d["failure"] == 1, "model"].value_counts().to_dict())

summary = pd.DataFrame({"drive_days": pd.Series(drive_days),
                        "failures": pd.Series(fails)}).fillna(0).astype(int)
print(summary.sort_values("failures", ascending=False).head(15))

91 files
                      drive_days  failures
HGST HUH721212ALN604      861297       180
TOSHIBA MG07ACA14TA      3368924       179
ST12000NM0008            1671057       145
ST8000NM0055             1177541       112
HGST HUH721212ALE604     1190182       110
TOSHIBA MG08ACA16TA      3614263       102
ST24000NM002H             861587        82
WDC WUH721816ALE6L4      2387389        74
TOSHIBA MG10ACA20TE      2117524        64
WDC WUH722222ALE6L4      4146339        63
ST16000NM001G            3162117        57
TOSHIBA MG08ACA16TEY      439243        46
ST12000NM001G            1193856        44
ST14000NM001G             949396        39
ST8000DM002               563210        38


In [8]:
import pandas as pd, glob

files = sorted(glob.glob("../data/raw/**/*.csv", recursive=True))
candidates = ["ST12000NM0008", "HGST HUH721212ALN604",
              "ST8000NM0055", "TOSHIBA MG07ACA14TA"]
sample_files = [files[0], files[45], files[-1]]

parts = []
for f in sample_files:
    for chunk in pd.read_csv(f, chunksize=100_000):
        parts.append(chunk[chunk["model"].isin(candidates)])
sample = pd.concat(parts)

smart_cols = [c for c in sample.columns
              if c.startswith("smart_") and c.endswith("_raw")]
fill = sample[smart_cols].notna().groupby(sample["model"]).mean()

for m in candidates:
    good = fill.loc[m][fill.loc[m] > 0.9]
    print(m, "->", len(good), "usable raw SMART columns")
    print(list(good.index), "\n")

ST12000NM0008 -> 23 usable raw SMART columns
['smart_1_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_18_raw', 'smart_187_raw', 'smart_188_raw', 'smart_190_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_195_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_200_raw', 'smart_240_raw', 'smart_241_raw', 'smart_242_raw'] 

HGST HUH721212ALN604 -> 18 usable raw SMART columns
['smart_1_raw', 'smart_2_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_8_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_22_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_196_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw'] 

ST8000NM0055 -> 24 usable raw SMART columns
['smart_1_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_184_raw', 'smart_187_raw', 'smart_188_raw', 'smart_189_raw',